MODELAMIENTO MACHINE LEARNING — ANÁLISIS PREDICTIVO DE CLIENTES

Dataset : base_estudio.xlsx

Período : 2024-01-02 → 2025-12-30

Corte   : historial 2024 → sep-2025 | target oct-dic 2025

ANÁLISIS INCLUIDOS
──────────────────
  
  A. CLASIFICACIÓN PREDICTIVA (3 modelos por problema)

       M1 — Churn           : ¿comprará en los próximos 90 días?
       M2 — Propensión Cat. : ¿qué categoría comprará?
       M3 — Valor Potencial : ¿a qué segmento de valor pertenece?


In [1]:
#from google.colab import drive
#drive.mount('/content/drive')

In [2]:
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import pickle
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist

from sklearn.cluster import KMeans
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (davies_bouldin_score, roc_curve, auc,
                              silhouette_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

from imblearn.over_sampling import SMOTE

from mlxtend.frequent_patterns import apriori, association_rules

In [3]:
# ── Carga de datos ────────────────────────────────────────────────────────────
#file_path = '/content/drive/MyDrive/MASTER/UPB/Semestre 1/Minería de Datos/Proyecto/base estudio.xlsx'
file_path = 'C:/Dev2/UPB/base estudio.xlsx'
df = pd.read_excel(file_path)

print(f"Dataset cargado con éxito: {df.shape[0]} filas y {df.shape[1]} columnas.")

Dataset cargado con éxito: 66974 filas y 11 columnas.


In [4]:
# ─────────────────────────────────────────────
# 0. CONFIGURACIÓN
# ─────────────────────────────────────────────
CUTOFF         = pd.Timestamp("2025-09-30")
TARGET_END     = pd.Timestamp("2025-12-30")
CV_FOLDS       = 5
RANDOM_SEED    = 42

In [5]:
# ─────────────────────────────────────────────
# 1. CARGA Y LIMPIEZA DE DATOS
# ─────────────────────────────────────────────
print("=" * 65)
print("1. CARGA Y EXPLORACIÓN DEL DATASET")
print("=" * 65)

df.columns = df.columns.str.strip()
df["Fecha"] = pd.to_datetime(df["Fecha"], format="%Y%m%d")
df = df.sort_values("Fecha").reset_index(drop=True)

print(f"  Shape total         : {df.shape}")
print(f"  Período             : {df['Fecha'].min().date()} → {df['Fecha'].max().date()}")
print(f"  Clientes únicos     : {df['ID cliente'].nunique()}")
print(f"  Nulos por columna   :\n{df.isnull().sum().to_string()}")
print(f"\n  Estadísticas Valor_bruto:\n{df['Valor_bruto'].describe().round(0).to_string()}")

1. CARGA Y EXPLORACIÓN DEL DATASET
  Shape total         : (66974, 11)
  Período             : 2024-01-02 → 2025-12-30
  Clientes únicos     : 60
  Nulos por columna   :
Fecha                           0
ID cliente                      0
Desc Ciudad  Cliente_factura    0
Criterio_mayor_item_5           0
Desc_criterio_mayor_item_5      0
Desc_criterio_mayor_item_2      0
Tipo de Documento               0
Numero_documento                0
Referencia Item                 0
Cantidad                        0
Valor_bruto                     0

  Estadísticas Valor_bruto:
count       66974.0
mean       413906.0
std        726705.0
min      -9089040.0
25%         89129.0
50%        175936.0
75%        410930.0
max      21536775.0


In [6]:
# ─────────────────────────────────────────────
# 2. SEPARACIÓN TEMPORAL (anti-leakage)
# Su objetivo principal es evitar el data leakage (fuga de datos), 
# asegurando que el modelo no "entrene con información del futuro".
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("2. SEPARACIÓN TEMPORAL — HISTORIAL / PERÍODO OBJETIVO")
print("=" * 65)

hist = df[df["Fecha"] <= CUTOFF].copy()
fut  = df[(df["Fecha"] > CUTOFF) & (df["Fecha"] <= TARGET_END)].copy()

print(f"  Historial : {hist.shape[0]:>6} filas | {hist['Fecha'].min().date()} → {hist['Fecha'].max().date()}")
print(f"  Futuro    : {fut.shape[0]:>6} filas | {fut['Fecha'].min().date()} → {fut['Fecha'].max().date()}")
print(f"  Clientes historial : {hist['ID cliente'].nunique()}")
print(f"  Clientes futuro    : {fut['ID cliente'].nunique()}")
print(f"  Clientes en ambos  : {len(set(hist['ID cliente']) & set(fut['ID cliente']))}")


2. SEPARACIÓN TEMPORAL — HISTORIAL / PERÍODO OBJETIVO
  Historial :  58428 filas | 2024-01-02 → 2025-09-30
  Futuro    :   8546 filas | 2025-10-01 → 2025-12-30
  Clientes historial : 60
  Clientes futuro    : 41
  Clientes en ambos  : 41


Rango Historial | 2024-01-02 a 2025-09-30 | Casi 21 meses de datos para "aprender" el comportamiento.

Rango Futuro | 2025-10-01 a 2025-12-30 | 3 meses para validar si las predicciones se cumplen.

Clientes en ambos | 41 | Este es el dato más importante. Tienes 41 clientes fieles de los cuales puedes estudiar su pasado para predecir su futuro.

Clientes perdidos | 19 (60 - 41) | Hay 19 clientes que estuvieron en el historial pero no realizaron acciones en el periodo futuro (posible churn o abandono).

In [7]:
# ─────────────────────────────────────────────
# 3. FEATURE ENGINEERING (solo desde historial)
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("3. FEATURE ENGINEERING (solo desde historial)")
print("=" * 65)

feats = hist.groupby("ID cliente").agg(
    total_transacciones     = ("Numero_documento",              "nunique"),
    total_items             = ("Referencia Item",               "count"),
    total_valor             = ("Valor_bruto",                   "sum"),
    avg_valor               = ("Valor_bruto",                   "mean"),
    std_valor               = ("Valor_bruto",                   "std"),
    max_valor               = ("Valor_bruto",                   "max"),
    total_cantidad          = ("Cantidad",                      "sum"),
    ciudades_distintas      = ("Desc Ciudad  Cliente_factura",  "nunique"),
    categorias_distintas    = ("Criterio_mayor_item_5",         "nunique"),
    subcategorias_distintas = ("Desc_criterio_mayor_item_2",    "nunique"),
    primera_compra          = ("Fecha",                         "min"),
    ultima_compra           = ("Fecha",                         "max"),
).reset_index()

feats["dias_activo"] = (feats["ultima_compra"] - feats["primera_compra"]).dt.days + 1
feats["recencia"]    = (CUTOFF - feats["ultima_compra"]).dt.days
feats["frecuencia"]  = feats["total_transacciones"] / (feats["dias_activo"] / 30)
feats["std_valor"]   = feats["std_valor"].fillna(0)

top_cat = (hist.groupby(["ID cliente", "Criterio_mayor_item_5"])
               .size().reset_index(name="cnt")
               .sort_values("cnt", ascending=False)
               .drop_duplicates("ID cliente")
               .rename(columns={"Criterio_mayor_item_5": "top_categoria"}))
feats = feats.merge(top_cat[["ID cliente", "top_categoria"]], on="ID cliente", how="left")

top_ciudad = (hist.groupby(["ID cliente", "Desc Ciudad  Cliente_factura"])
                  .size().reset_index(name="cnt")
                  .sort_values("cnt", ascending=False)
                  .drop_duplicates("ID cliente")
                  .rename(columns={"Desc Ciudad  Cliente_factura": "top_ciudad"}))
feats = feats.merge(top_ciudad[["ID cliente", "top_ciudad"]], on="ID cliente", how="left")

print(f"  Variables generadas : {feats.shape[1] - 1} para {feats.shape[0]} clientes")


3. FEATURE ENGINEERING (solo desde historial)
  Variables generadas : 17 para 60 clientes


1. Agregación Principal (El "Perfil" del Cliente)
La función .agg() compacta miles de filas en una sola por cada ID cliente. Crea métricas de tres tipos:

* Monetarias: total_valor, avg_valor, max_valor (¿Cuánto dinero gasta?).

* Volumen: total_items, total_cantidad (¿Qué tanto compra?).

* Diversidad: ciudades_distintas, categorias_distintas (¿Qué tan variado es su comportamiento?).

* Lealtad: primera_compra y ultima_compra.



2. Creación de Métricas RFM y de Tiempo
Aquí el código calcula variables más "inteligentes" derivadas de las fechas:

* dias_activo: Tiempo total desde que el cliente nos conoce hasta su última compra registrada.

* recencia: Días que han pasado desde su última compra hasta el CUTOFF. Si este número es muy alto, el cliente podría estar por irse.

* frecuencia: Normaliza las compras. No es lo mismo 10 compras en 10 días que 10 compras en 2 años. Aquí se mide como "compras por mes".

* std_valor: La desviación estándar. Si es alta, el cliente es irregular (a veces compra mucho y a veces poco); si es 0 (usando .fillna(0)), es que solo ha comprado una vez o siempre gasta lo mismo.

3. Identificación de Preferencias (top_cat y top_ciudad). El código busca "el favorito" de cada cliente:

* Agrupa por cliente y categoría/ciudad.

* Cuenta cuántas veces aparece cada una.

* Ordena de mayor a menor y usa .drop_duplicates("ID cliente") para quedarse solo con la más frecuente.

* Merge: Pega esa "Categoría Favorita" y "Ciudad Principal" a la tabla de características.

In [8]:
feats.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   ID cliente               60 non-null     int64         
 1   total_transacciones      60 non-null     int64         
 2   total_items              60 non-null     int64         
 3   total_valor              60 non-null     int64         
 4   avg_valor                60 non-null     float64       
 5   std_valor                60 non-null     float64       
 6   max_valor                60 non-null     int64         
 7   total_cantidad           60 non-null     int64         
 8   ciudades_distintas       60 non-null     int64         
 9   categorias_distintas     60 non-null     int64         
 10  subcategorias_distintas  60 non-null     int64         
 11  primera_compra           60 non-null     datetime64[us]
 12  ultima_compra            60 non-null     datetime

In [9]:
# ─────────────────────────────────────────────
# 4. CONSTRUCCIÓN DE TARGETS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("4. CONSTRUCCIÓN DE TARGETS (período futuro)")
print("=" * 60)


4. CONSTRUCCIÓN DE TARGETS (período futuro)


* feats["ID cliente"].isin(clientes_activos_fut): Marca con True a los clientes que sí volvieron a comprar.

    *  El símbolo ~ (NOT): Invierte el resultado. Ahora, los que no volvieron a comprar son True.

* .astype(int): Convierte los valores booleanos en números:

    * 1 (Churn): El cliente estuvo en el historial pero no apareció en el futuro (se fugó).
    * 0 (No Churn): El cliente estuvo en el historial y sí volvió a aparecer en el futuro (fiel)

In [10]:
# — M1: Churn binario —
clientes_activos_fut = fut["ID cliente"].unique() # Crea una lista de todos los IDs de clientes que realizaron al menos una transacción en el periodo futuro (fut)
feats["churn"] = (~feats["ID cliente"].isin(clientes_activos_fut)).astype(int)
print(f"  M1 Churn     → {feats['churn'].value_counts().to_dict()}")

  M1 Churn     → {0: 41, 1: 19}


1. Identificar la "Categoría Favorita" en el Futuro

* El código analiza el periodo futuro (fut) para ver qué compró cada cliente. Si un cliente compró 5 veces "Herramientas" y 2 veces "Pinturas", el código determina que su cat_futura es "Herramientas".

    * Agrupa: Por cliente y categoría.
    * Ordena y Limpia: Se queda solo con la categoría más frecuente por cada ID.

2. Integración y Manejo de los que no Compraron

* Se une esta información a nuestra tabla maestra de características (feats).

    * fillna("NO_COMPRA"): Es crucial. Los clientes que estaban en el historial pero no compraron nada en el futuro (los que marcamos como Churn en el paso anterior) reciben la etiqueta "NO_COMPRA".

3. Agrupación de Categorías Minoritarias

Si hay 50 categorías diferentes, el modelo se volvería loco intentando predecir clases con muy pocos ejemplos. Para simplificar el problema:

    * Identifica las 6 más populares (incluyendo probablemente "NO_COMPRA").

    * Agrupa el resto: Cualquier categoría que no esté en el "Top 6" se renombra como "OTROS".


In [11]:
# — M2: Categoría top en futuro (multiclase) —
fut_cat = (fut.groupby(["ID cliente", "Desc_criterio_mayor_item_5"])
              .size().reset_index(name="cnt")
              .sort_values("cnt", ascending=False)
              .drop_duplicates("ID cliente")
              .rename(columns={"Desc_criterio_mayor_item_5": "cat_futura"}))
feats = feats.merge(fut_cat[["ID cliente", "cat_futura"]], on="ID cliente", how="left")
feats["cat_futura"] = feats["cat_futura"].fillna("NO_COMPRA")
top_cats_fut = feats["cat_futura"].value_counts().head(6).index
feats["cat_futura_top"] = feats["cat_futura"].apply(
    lambda x: x if x in top_cats_fut else "OTROS"
)
print(f"  M2 Categoría → {feats['cat_futura_top'].value_counts().to_dict()}")

  M2 Categoría → {'PINTUCO': 21, 'NO_COMPRA': 19, 'REVESTIMIENTO CORONA': 6, 'PLOMERIA GRIVAL': 4, 'OTROS': 4, 'ACCESORIOS GERFOR': 4, 'ICO': 2}


1. Cálculo del Gasto Futuro Total

* Suma las ventas: Agrupa todas las transacciones del periodo futuro por cliente y suma el Valor_bruto.

* Merge y limpieza: Une este dato a tu tabla principal. Los clientes que no compraron nada en el futuro aparecen como NaN tras el merge, por lo que se usa .fillna(0) para decir que su gasto fue de $0.

2. Definición de los Límites (Cuantiles)

* calcula los segmentos basándose en la gente que sí compró.

* Si se incluye a los que gastaron $0, los resultados estarían sesgados.

* Utiliza los tertiles (33% y 66%) para dividir a los compradores en tres grupos de igual tamaño: los que gastan poco, los que gastan promedio y los que más aportan (Top 33%)

3. Función de Segmentación

La función segmentar(v) crea las etiquetas finales:

No_Compra: Gasto exactamente igual a 0.

Bajo: El 33% de los compradores con menor gasto.

Medio: El 33% intermedio.

Alto: Los "VIP" o clientes de alto valor (el 33% que más gastó).

In [12]:
# — M3: Segmento de valor futuro (cuartiles) —
fut_valor = (fut.groupby("ID cliente")["Valor_bruto"]
                .sum().reset_index(name="valor_futuro"))
feats = feats.merge(fut_valor, on="ID cliente", how="left")
feats["valor_futuro"] = feats["valor_futuro"].fillna(0)

compradores = feats[feats["valor_futuro"] > 0]["valor_futuro"]
q1, q2 = compradores.quantile([0.33, 0.66]).values

def segmentar(v):
    if v == 0:    return "No_Compra"
    elif v <= q1: return "Bajo"
    elif v <= q2: return "Medio"
    else:         return "Alto"

feats["segmento_valor"] = feats["valor_futuro"].apply(segmentar)
print(f"M3 Segmento → {feats['segmento_valor'].value_counts().to_dict()}")

M3 Segmento → {'No_Compra': 19, 'Alto': 14, 'Bajo': 14, 'Medio': 13}


1. Encoding (Codificación de Variables Categóricas)

LabelEncoder: Asigna un número único a cada categoría (ej: Medellín = 0, Bogotá = 1, Cali = 2).

.fillna("DESCONOCIDO"): trata los valores nulos antes de codificar, para que el modelo sepa que la "ausencia de datos" también es una categoría válida.


2. Definición de la Matriz de Características ($X$)

* Aquí selecciona solo las columnas que servirán como predictores. solo incluye las variables del pasado (calculadas en el paso de Feature Engineering).

Importante: No incluye churn, cat_futura ni valor_futuro aquí, porque esas son las respuestas que el modelo debe adivinar.

Se aplica un .fillna(0) final por seguridad para asegurar que no haya huecos en la matriz matemática.

3. Análisis de Correlación

El código calcula qué tanto "se mueve" una variable cuando el Churn cambia.

.corr(): Calcula el coeficiente de correlación de Pearson (que va de -1 a 1).

Significado de los valores:

* Cercano a 1: Correlación positiva fuerte. Si la variable sube, el riesgo de Churn sube. (Ejemplo típico: recencia).

* Cercano a -1: Correlación negativa fuerte. Si la variable sube, el Churn baja. (Ejemplo típico: total_valor o frecuencia).

* Cercano a 0: La variable no ayuda mucho a predecir la fuga.

4. Interpretación del Output

* El código imprime las 8 variables que más influyen en el Churn ordenadas por importancia absoluta.

* Si la recencia tiene un valor de 0.65, confirmas que mientras más días pasan sin que el cliente compre, es mucho más probable que se fugue.

* Si ves que total_transacciones tiene -0.40, confirmas que los clientes que más veces compran son los más leales.

In [13]:
# ─────────────────────────────────────────────
# 5. PREPARACIÓN DE FEATURES PARA MODELADO
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("5. PREPARACIÓN DE FEATURES — ENCODING Y CORRELACIONES")
print("=" * 65)

le_cat = LabelEncoder()
le_ciu = LabelEncoder()
feats["top_categoria_enc"] = le_cat.fit_transform(feats["top_categoria"].fillna("DESCONOCIDO"))
feats["top_ciudad_enc"]    = le_ciu.fit_transform(feats["top_ciudad"].fillna("DESCONOCIDO"))

NUM_COLS = [
    "total_transacciones", "total_items", "total_valor", "avg_valor", "std_valor",
    "max_valor", "total_cantidad", "ciudades_distintas", "categorias_distintas",
    "subcategorias_distintas", "dias_activo", "recencia", "frecuencia",
    "top_categoria_enc", "top_ciudad_enc",
]
X = feats[NUM_COLS].fillna(0)

print("  Top correlaciones con target Churn:")
corr_churn = pd.concat([X, feats["churn"]], axis=1).corr()["churn"].drop("churn")
print(corr_churn.sort_values(key=abs, ascending=False).head(8).round(4).to_string())


5. PREPARACIÓN DE FEATURES — ENCODING Y CORRELACIONES
  Top correlaciones con target Churn:
recencia                   0.7775
categorias_distintas      -0.6534
dias_activo               -0.6505
subcategorias_distintas   -0.6482
total_valor               -0.5074
max_valor                 -0.4996
total_cantidad            -0.4564
total_transacciones       -0.3817


1. El problema del desbalance

Como vimos antes, la distribución es aproximadamente 2:1. SMOTE lo que hace es fabricar ejemplos nuevos de la clase minoritaria (los que se fugan) para que ambas clases tengan el mismo peso (50/50).

2. La lógica del código

* y_churn = feats["churn"]: Define la variable objetivo (quién se fugó y quién no).

* k_nbrs = min(5, y_churn.value_counts().min() - 1): Esta línea es un ajuste de seguridad. SMOTE crea datos nuevos mirando a los "vecinos" más cercanos de cada punto. Como hay pocos datos, el código se asegura de no buscar más vecinos de los que realmente existen en la clase minoritaria para evitar errores.

* SMOTE(...): Configura el algoritmo con una semilla aleatoria (RANDOM_SEED) para que los resultados sean replicables.

* sm.fit_resample(X, y_churn): El algoritmo crea registros sintéticos (falsos pero estadísticamente coherentes) y devuelve:

    * X_bal: Una nueva matriz de características con más filas.

    * y_bal: Una nueva lista de etiquetas balanceada.

In [14]:
# ─────────────────────────────────────────────
# 6. BALANCEO — SMOTE (solo M1)
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("6. BALANCEO CON SMOTE — M1 Churn")
print("=" * 65)

y_churn = feats["churn"]
k_nbrs  = min(5, y_churn.value_counts().min() - 1)
sm      = SMOTE(random_state=RANDOM_SEED, k_neighbors=k_nbrs)
X_bal, y_bal = sm.fit_resample(X, y_churn)
print(f"  Antes : {dict(y_churn.value_counts())}")
print(f"  Después SMOTE : {dict(pd.Series(y_bal).value_counts())}")


6. BALANCEO CON SMOTE — M1 Churn
  Antes : {0: np.int64(41), 1: np.int64(19)}
  Después SMOTE : {0: np.int64(41), 1: np.int64(41)}


1. Modelos (get_models)

La función devuelve un diccionario con tres algoritmos de naturalezas muy distintas.

* Logistic Regression: Es un modelo lineal. Se usa un Pipeline con StandardScaler porque este algoritmo es sensible a la escala de los números (necesita que el "gasto total" y la "recencia" hablen el mismo idioma numérico).

* Random Forest: Es un conjunto de árboles de decisión. Es excelente para captar relaciones no lineales y, al usar class_weight="balanced", le da un empujón extra de importancia a la clase minoritaria (los que se fugan).

* Gradient Boosting: Otro modelo basado en árboles, pero que aprende de sus propios errores de forma secuencial. Suele ser el más preciso, aunque es más propenso al sobreajuste (overfitting) si no se controla bien.

* KNN: es un modelo basado en distancias (calcula qué tan "cerca" está un cliente de otros en un espacio n-dimensional). Si un nuevo cliente tiene una recencia, frecuencia y gasto muy similares a otros 5 clientes que se fugaron (churn=1), el modelo clasificará a este nuevo cliente también como fuga.

2. Validación Cruzada Estratificada (StratifiedKFold)

Asegurar que los resultados sean reales y no "suerte":

* n_splits=CV_FOLDS: Divide tus datos en varias partes (por ejemplo, 5). Entrena el modelo en 4 partes y lo prueba en la restante, repitiendo el proceso hasta que todos los datos hayan sido "el examen" una vez.

* Stratified (Estratificada): Esto es vital dado que hay pocos datos. Asegura que en cada "pedazo" de la división se mantenga la proporción original de clientes que se fugan y clientes que se quedan. Sin esto, podría terminar con un grupo de prueba que no tenga ningún cliente fugado, haciendo que la evaluación sea inútil.

In [15]:
# ─────────────────────────────────────────────
# 7. DEFINICIÓN DE MODELOS (4 por problema)
# ─────────────────────────────────────────────
def get_models():
    return {
        "Logistic Regression": Pipeline([
            ("sc", StandardScaler()),
            ("m",  LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
        ]),
        "Random Forest": RandomForestClassifier(
            n_estimators=200, random_state=RANDOM_SEED, class_weight="balanced"
        ),
        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=200, random_state=RANDOM_SEED
        ),
        "KNN": Pipeline([
            ("sc", StandardScaler()),
            ("m",  KNeighborsClassifier(n_neighbors=5))
        ]),
    }

cv_strat = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
results   = {}

1. Bucle de Validación Cruzada

El código recorre cada modelo (Regresión Logística, Random Forest, Gradient Boosting y KNN) y los somete a una prueba de estrés:

* cross_validate: No entrena el modelo una sola vez. Lo entrena y prueba múltiples veces (según cv_strat) usando diferentes porciones de tus datos balanceados (X_bal, y_bal).

* Métricas de evaluación:

    * Accuracy: Porcentaje total de aciertos. (Cuidado: puede ser engañoso).

    * F1-Score: El balance entre Precisión (no dar falsas alarmas) y Recall (no olvidar a ningún fugado). Es la métrica más honesta para este problema.

    * ROC-AUC: Indica qué tan bien el modelo separa a los clientes "fieles" de los "fugados". Un 1.0 es perfecto, un 0.5 es como lanzar una moneda.

2. Almacenamiento y Selección del Ganador

El código guarda los promedios de las métricas de cada modelo en un diccionario.

La estrella (★): Al final, el código busca automáticamente quién tuvo el F1-Score más alto y lo declara el ganador oficial del problema de Churn.

3. Importancia de Variables

* model_final_churn = RandomForestClassifier(...)
* model_final_churn.fit(X_bal, y_bal)

Después de encontrar al ganador, el código entrena un Random Forest por separado con el fin de extraer la "importancia de las características" (Feature Importance).

Los modelos basados en árboles (como Random Forest) son excelentes para saber qué columnas fueron las más útiles para tomar la decisión. Por ejemplo, permitirá saber si la Recencia (días desde la última compra) fue más determinante que el Total Valor para predecir la fuga.

In [16]:
# ─────────────────────────────────────────────
# M1 — CHURN (binario)
# ─────────────────────────────────────────────
print("\n  [M1] Churn — Clasificación Binaria")

results["M1_Churn"] = {}

for name, model in get_models().items():
    sc = cross_validate(model, X_bal, y_bal, cv=cv_strat,
                        scoring=["accuracy", "f1", "roc_auc"])
    results["M1_Churn"][name] = {
        "accuracy": round(sc["test_accuracy"].mean(), 4),
        "f1":       round(sc["test_f1"].mean(),       4),
        "roc_auc":  round(sc["test_roc_auc"].mean(),  4),
        "f1_std":   round(sc["test_f1"].std(),        4),
    }
    print(f"    {name:<22} Acc={results['M1_Churn'][name]['accuracy']} "
          f"F1={results['M1_Churn'][name]['f1']} "
          f"AUC={results['M1_Churn'][name]['roc_auc']}")
best_m1 = max(results["M1_Churn"], key=lambda k: results["M1_Churn"][k]["f1"])
print(f"    ★ Mejor M1 : {best_m1}")

# Feature importance
#rf_fi 
model_final_churn = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, class_weight="balanced")
model_final_churn.fit(X_bal, y_bal)


  [M1] Churn — Clasificación Binaria


    Logistic Regression    Acc=0.9757 F1=0.9761 AUC=0.9826
    Random Forest          Acc=0.9265 F1=0.9289 AUC=0.9771
    Gradient Boosting      Acc=0.9265 F1=0.9095 AUC=0.975
    KNN                    Acc=0.8654 F1=0.8708 AUC=0.9568
    ★ Mejor M1 : Logistic Regression


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

1. Preparación del Target Multiclase

* le_m2 = LabelEncoder()
* y_cat = le_m2.fit_transform(feats["cat_futura_top"])

Como las categorías son nombres (ej: "Herramientas", "Hogar", "NO_COMPRA"), el LabelEncoder las convierte en números (0, 1, 2, 3...). Esto es necesario para que los algoritmos puedan realizar los cálculos de probabilidad para cada clase.

2. Modelos (M2)

El código vuelve a recorrer los modelos (get_models()), pero con dos diferencias clave respecto al modelo de Churn:

* Usa X (Original): Aquí no se usa X_bal (SMOTE), sino los datos originales. Esto es porque SMOTE es sencillo para 2 clases, pero mucho más complejo para problemas multiclase con categorías muy variadas.

* Métricas Específicas:

    * f1_weighted: En lugar de un f1 simple, utiliza la versión "ponderada". Esto es vital porque algunas categorías se compran mucho más que otras. Esta métrica calcula el F1 para cada categoría y saca un promedio basado en el número de ejemplos de cada una, evitando que las clases pequeñas arruinen el puntaje.

3. Evaluación de Estabilidad

Se calcula la desviación estándar del F1-Score. Esto sirve para saber si el modelo es "estable".

* Std baja: El modelo rinde igual de bien con cualquier grupo de clientes.

* Std alta: El modelo es inestable; a veces acierta mucho y a veces falla estrepitosamente dependiendo de los datos que le toquen.

4. Selección del Ganador

Si el Mejor M2 tiene un F1w alto (por ejemplo, arriba de 0.70), significa que es una herramienta muy poderosa para:

* Cross-selling: Si el modelo dice que un cliente de "Construcción" va a comprar "Pinturas", puedes enviarle una oferta justo antes.

* Personalización: Puedes cambiar el contenido de los correos electrónicos o de la app para cada cliente basándote en la categoría que el modelo prediga.

In [17]:
# ─────────────────────────────────────────────
# M2 — PROPENSIÓN CATEGORÍA (multiclase)
# ─────────────────────────────────────────────
print("\n  [M2] Propensión Categoría — Multiclase")

le_m2 = LabelEncoder()
y_cat = le_m2.fit_transform(feats["cat_futura_top"])

results["M2_Categoria"] = {}

for name, model in get_models().items():
    sc = cross_validate(model, X, y_cat, cv=cv_strat,
                        scoring=["accuracy", "f1_weighted"])
    results["M2_Categoria"][name] = {
        "accuracy":    round(sc["test_accuracy"].mean(),    4),
        "f1_weighted": round(sc["test_f1_weighted"].mean(), 4),
        "f1_std":      round(sc["test_f1_weighted"].std(),  4),
    }
    print(f"    {name:<22} Acc={results['M2_Categoria'][name]['accuracy']} "
          f"F1w={results['M2_Categoria'][name]['f1_weighted']}")
best_m2 = max(results["M2_Categoria"], key=lambda k: results["M2_Categoria"][k]["f1_weighted"])
print(f"    ★ Mejor M2 : {best_m2}")

model_final_cat = GradientBoostingClassifier(n_estimators=200, random_state=RANDOM_SEED)
model_final_cat.fit(X, y_cat)


  [M2] Propensión Categoría — Multiclase
    Logistic Regression    Acc=0.5667 F1w=0.5107
    Random Forest          Acc=0.6167 F1w=0.5291
    Gradient Boosting      Acc=0.6167 F1w=0.5817
    KNN                    Acc=0.5667 F1w=0.4652
    ★ Mejor M2 : Gradient Boosting


,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are'friedman_mse' for the mean squared error with improvement score byFriedman, 'squared_error' for mean squared error. The default value of'friedman_mse' is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, 

1. Preparación de la "Variable de Valor"
* le_m3 = LabelEncoder()
* y_seg = le_m3.fit_transform(feats["segmento_valor"])

El LabelEncoder traduce tus etiquetas de texto a números. Internamente, transformará algo como:
* Alto $\rightarrow$ 0
* Bajo $\rightarrow$ 1
* Medio $\rightarrow$ 2
* No_Compra $\rightarrow$ 3

2. Modelos (M3)

El código somete a los modelos a la misma validación cruzada estratificada para asegurar que en cada prueba haya representación de clientes de alto, medio y bajo valor.

* Métricas utilizadas:

    * Accuracy: ¿Qué tan cerca estuvo de asignarle el presupuesto correcto a cada cliente?

    * F1-weighted: Es la métrica decisiva aquí. Como es más difícil predecir quién será un cliente de "Alto Valor" (porque suelen ser pocos), el F1 ponderado castiga al modelo si falla en esos segmentos críticos aunque acierte en los clientes que no compran.

3. ¿Por qué es importante este modelo (M3)?

    Si el modelo M1 te dice quién se va, y el M2 te dice qué comprarán los que se quedan, el M3 te dice cuánto dinero está en juego.

    Este modelo permite aplicar una estrategia de "Value-Based Marketing":

    * Priorización: Si el modelo predice que un cliente será de segmento "Alto", el equipo comercial debería llamarlo personalmente.

    * Optimización de recursos: Si el modelo predice que un cliente es de segmento "Bajo", quizás baste con un correo automatizado.

In [18]:
# ─────────────────────────────────────────────
# M3 — VALOR POTENCIAL (multiclase)
# ─────────────────────────────────────────────
print("\n  [M3] Segmento Valor Futuro — Multiclase")

le_m3 = LabelEncoder()
y_seg = le_m3.fit_transform(feats["segmento_valor"])

results["M3_Segmento"] = {}

for name, model in get_models().items():
    sc = cross_validate(model, X, y_seg, cv=cv_strat,
                        scoring=["accuracy", "f1_weighted"])
    results["M3_Segmento"][name] = {
        "accuracy":    round(sc["test_accuracy"].mean(),    4),
        "f1_weighted": round(sc["test_f1_weighted"].mean(), 4),
        "f1_std":      round(sc["test_f1_weighted"].std(),  4),
    }
    print(f"    {name:<22} Acc={results['M3_Segmento'][name]['accuracy']} "
          f"F1w={results['M3_Segmento'][name]['f1_weighted']}")
best_m3 = max(results["M3_Segmento"], key=lambda k: results["M3_Segmento"][k]["f1_weighted"])
print(f"    ★ Mejor M3 : {best_m3}")

model_final_valor = Pipeline([
    ("sc", StandardScaler()),
    ("m",  LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)),
])
model_final_valor.fit(X, y_seg)


  [M3] Segmento Valor Futuro — Multiclase
    Logistic Regression    Acc=0.65 F1w=0.6341
    Random Forest          Acc=0.6333 F1w=0.6139
    Gradient Boosting      Acc=0.55 F1w=0.537
    KNN                    Acc=0.5333 F1w=0.5179
    ★ Mejor M3 : Logistic Regression


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('sc', ...), ('m', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with 

In [19]:
# Nombre del archivo
filename = 'modelo_360.pkl'

# Extraer los nombres de las columnas
variables = X.columns.tolist()

# Empaquetar todo usando los nombres de las variables del paso anterior
paquete_modelos = {
    "modelo_churn": model_final_churn, # El Random Forest de M1
    "modelo_cat":   model_final_cat,   # El Gradient Boosting de M2
    "modelo_valor": model_final_valor, # El Pipeline de Regresión de M3
    "le_m2":        le_m2,             # Encoder de M2
    "le_m3":        le_m3,             # Encoder de M3
    "variables":    variables          # Lista de columnas
}

# Guardar en disco
with open(filename, 'wb') as file:
    pickle.dump(paquete_modelos, file)

In [20]:
# ─────────────────────────────────────────────────────────────
# 14. RESUMEN FINAL
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("14. RESUMEN FINAL — MEJOR MODELO / HALLAZGO POR ANÁLISIS")
print("=" * 65)

best_m1 = max(results["M1_Churn"],     key=lambda k: results["M1_Churn"][k]["f1"])
best_m2 = max(results["M2_Categoria"], key=lambda k: results["M2_Categoria"][k]["f1_weighted"])
best_m3 = max(results["M3_Segmento"],  key=lambda k: results["M3_Segmento"][k]["f1_weighted"])

print(f"\n  A. CLASIFICACIÓN")
print(f"     M1 Churn     ★ {best_m1:<24} "
      f"Acc={results['M1_Churn'][best_m1]['accuracy']} "
      f"F1={results['M1_Churn'][best_m1]['f1']} "
      f"AUC={results['M1_Churn'][best_m1]['roc_auc']}")
print(f"     M2 Categ.    ★ {best_m2:<24} "
      f"Acc={results['M2_Categoria'][best_m2]['accuracy']} "
      f"F1w={results['M2_Categoria'][best_m2]['f1_weighted']}")
print(f"     M3 Segmento  ★ {best_m3:<24} "
      f"Acc={results['M3_Segmento'][best_m3]['accuracy']} "
      f"F1w={results['M3_Segmento'][best_m3]['f1_weighted']}")


14. RESUMEN FINAL — MEJOR MODELO / HALLAZGO POR ANÁLISIS

  A. CLASIFICACIÓN
     M1 Churn     ★ Logistic Regression      Acc=0.9757 F1=0.9761 AUC=0.9826
     M2 Categ.    ★ Gradient Boosting        Acc=0.6167 F1w=0.5817
     M3 Segmento  ★ Logistic Regression      Acc=0.65 F1w=0.6341


Modelamiento — Resultados

Preparación y Separación Temporal (Anti-Leakage)
El dataset tiene 66.974 transacciones, 60 clientes, período 2024-01-02 a 2025-12-30. Para evitar el leakage que generaba F1=1.0, se aplicó un corte temporal estricto:

* Historial (features): 2024-01 → sep-2025 (58.428 filas)

* Período objetivo (target): oct-dic 2025 (8.546 filas)

* 41 clientes compraron en el futuro; 19 no (churn real)

Se construyeron 15 variables de comportamiento histórico: recencia, frecuencia, valor total/promedio/máx, nº transacciones, diversidad de categorías y ciudades, días activo, entre otras. Se aplicó SMOTE para balancear el target de churn (41 vs 19 clientes).